In [1]:
# Import necessary libraries
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib

# Load and preprocess MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = np.expand_dims(x_train, -1)  # Shape: (samples, 28, 28, 1)
x_test = np.expand_dims(x_test, -1)

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# Define CNN model using the Functional API
inputs = Input(shape=(28, 28, 1))
x = Conv2D(32, (3, 3), activation='relu')(inputs)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(10, activation='softmax')(x)

cnn_model = Model(inputs=inputs, outputs=outputs)
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

# Train CNN model
cnn_model.fit(x_train, y_train_cat, epochs=5, batch_size=128, validation_data=(x_test, y_test_cat))

# Create feature extractor model (outputs 128-d feature vector before dropout and final layer)
feature_extractor = Model(inputs=cnn_model.input, outputs=cnn_model.layers[-3].output)

# Extract features from train and test sets using CNN feature extractor
train_features = feature_extractor.predict(x_train)
test_features = feature_extractor.predict(x_test)
feature_extractor.save('featureextractor.h5')

# Train SVM classifier on CNN features
svm_model = SVC(kernel='linear')
svm_model.fit(train_features, y_train)

# Evaluate SVM classifier on test features
y_pred = svm_model.predict(test_features)
accuracy = accuracy_score(y_test, y_pred)
print(f"Hybrid CNN-SVM accuracy: {accuracy:.4f}")

# Save the trained SVM model
joblib.dump(svm_model, 'svm_model.pkl')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9073 - loss: 0.3034 - val_accuracy: 0.9804 - val_loss: 0.0620
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9712 - loss: 0.0981 - val_accuracy: 0.9857 - val_loss: 0.0434
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9788 - loss: 0.0716 - val_accuracy: 0.9882 - val_loss: 0.0335
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9828 - loss: 0.0590 - val_accuracy: 0.9889 - val_loss: 0.0319
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9851 - loss: 0.0492 - val_accuracy: 0.9905 - val_loss: 0.0278
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


Hybrid CNN-SVM accuracy: 0.9919


['svm_model.pkl']

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install Flask

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from sklearn.metrics import accuracy_score
import joblib

# Load MNIST test data and preprocess
(_, _), (x_test, y_test) = mnist.load_data()
x_test = x_test.astype('float32') / 255.
x_test = np.expand_dims(x_test, -1)  # Shape: (num_samples, 28, 28, 1)

# Re-create feature extractor model (output before dropout and last Dense layer)
feature_extractor = Model(inputs=cnn_model.input, outputs=cnn_model.layers[-3].output)

# Extract features from test data
test_features = feature_extractor.predict(x_test)

# Load saved SVM model (replace 'svm_model.pkl' with your filename)
svm_model = joblib.load('svm_model.pkl')

# Predict using SVM
y_pred = svm_model.predict(test_features)

# Calculate and print test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Hybrid CNN-SVM test accuracy: {accuracy:.4f}")

# Print sample predictions
for i in range(10):
    print(f"True: {y_test[i]}, Predicted: {y_pred[i]}")


In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

# The uploaded file name is now available in the 'uploaded' dictionary
# You can access it like this: list(uploaded.keys())[0]

In [ ]:
!pip install opencv-python

# Task
Classify the digit in the uploaded image "image.jpeg" using the trained CNN feature extractor and SVM classifier.

## Load and preprocess the image

### Subtask:
Load the uploaded JPEG image, convert it to grayscale, and resize it to 28x28 pixels.


**Reasoning**:
Load the uploaded image, convert it to grayscale, and resize it to 28x28 pixels using OpenCV.



## Normalize the image

### Subtask:
Scale the pixel values to be between 0 and 1, similar to how the training data was preprocessed.


**Reasoning**:
Convert the resized image to float32 and scale the pixel values to be between 0 and 1.



In [ ]:
# Convert to float32 and scale pixel values to 0-1
normalized_image = resized_image.astype('float32') / 255.0

# Display the shape and some pixel values of the normalized image to verify
print(f"Shape of the normalized image: {normalized_image.shape}")
print(f"Sample pixel values of the normalized image: {normalized_image[0, :5]}")

## Expand dimensions

### Subtask:
Add an extra dimension to the image data to match the input shape of the feature extractor model.


**Reasoning**:
Add an extra dimension to the normalized image to match the input shape of the feature extractor model.



In [ ]:
# Add an extra dimension to the image data to match the input shape of the feature extractor model
input_image_for_cnn = np.expand_dims(normalized_image, -1)

# Print the shape of the input_image_for_cnn to verify the added dimension
print(f"Shape of the input image for CNN: {input_image_for_cnn.shape}")

## Extract features

### Subtask:
Use the trained CNN feature extractor model to get the feature vector for the preprocessed image.


**Reasoning**:
Use the trained CNN feature extractor model to get the feature vector for the preprocessed image and store it in a variable named `image_features`, then print the shape of `image_features`.



In [ ]:
# Use the trained CNN feature extractor model to get the feature vector
image_features = feature_extractor.predict(np.expand_dims(input_image_for_cnn, axis=0))

# Print the shape of image_features to verify the output dimension
print(f"Shape of the extracted image features: {image_features.shape}")

## Predict with svm

### Subtask:
Use the loaded SVM model to predict the digit based on the extracted features.


**Reasoning**:
Use the loaded SVM model to predict the digit based on the extracted features.



In [ ]:
# Use the loaded SVM model to predict the digit based on the extracted features
predicted_digit = svm_model.predict(image_features)

# Print the predicted digit
print(f"Predicted digit: {predicted_digit[0]}")

## Display the prediction

### Subtask:
Show the predicted digit to the user.


**Reasoning**:
Print the predicted digit in a user-friendly format.



In [ ]:
# Print a user-friendly message indicating the predicted digit.
print(f"The predicted digit is: {predicted_digit[0]}")

## Summary:

### Data Analysis Key Findings

*   The uploaded image was successfully preprocessed by converting it to grayscale and resizing it to 28x28 pixels.
*   The pixel values of the image were normalized to a range between 0 and 1.
*   An extra dimension was added to the image data, changing its shape to (28, 28, 1) to match the expected input of the CNN feature extractor.
*   The trained CNN feature extractor successfully extracted a feature vector of size 128 from the preprocessed image.
*   The loaded SVM model predicted the digit as 8 based on the extracted features.

### Insights or Next Steps

*   The combination of a CNN feature extractor and an SVM classifier appears to be effective for this digit classification task.
*   To improve performance, consider exploring different CNN architectures, fine-tuning the feature extractor, or experimenting with other classification algorithms.
